# Домашнее задание № 1

## Задание 1 (2 балла)

Найдите 1 любой способ сломать токенизацию на предложения функцией sentenize из библиотеки razdel. Придумайте (или найдите на каком-то корпусе) такое предложение (или несколько предложений), которое будет некорректно разобрано sentenize, но при этом будет грамматически корректным. 

In [1]:
%pip install -q razdel pymystem3 pymorphy3 spacy
%pip install -q https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.8.0/ru_core_news_sm-3.8.0-py3-none-any.whl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from razdel import sentenize

candidates = [
    "Он сказал: «Приду завтра. Может быть, раньше». Я согласился.",
    "Это стоит 5.6.7 рублей. Дорого.",
    "Он родился в 1990 г. в Москве. Потом уехал.",
    "См. рис. 5. Там всё видно.",
    "Съездил в г. Москва. Вернулся.",
    "Заработал 100 руб. за день. Мало.",
    "Пришёл в 5.30 утра. Мы ждали.",
    "А. С. Пушкин родился в Москве. Его произведения известны.",
]

found_broken = False
for ex in candidates:
    sents = [s.text for s in sentenize(ex)]
    if len(sents) > 2:
        found_broken = True
        print("исходник:", ex)
        for i, s in enumerate(sents, 1):
            print(f"  {i}. {s!r}")
        print("sentenize разрезал больше, чем нужно")

исходник: Он сказал: «Приду завтра. Может быть, раньше». Я согласился.
  1. 'Он сказал: «Приду завтра.'
  2. 'Может быть, раньше».'
  3. 'Я согласился.'
sentenize разрезал больше, чем нужно


## 2. Токенизация Mystem vs razdel.tokenize (2 балла)


Токенизируйте текст с помощью razdel и с помощью Mystem. Найдите различия в токенизациях. Что по вашему работает лучше на приведенном тексте?

In [5]:
text = """
Вторым и третьим открытыми белыми карликами стали Сириус B и Процион B. В 1844 году директор Кёнигсбергской обсерватории Фридрих Бессель, анализируя данные наблюдений, которые велись с 1755 года, обнаружил, что Сириус, ярчайшая звезда земного неба, и Процион периодически, хотя и весьма слабо, отклоняются от прямолинейной траектории движения по небесной сфере[5]. Бессель пришёл к выводу, что у каждой из них должен быть близкий спутник. Сообщение было встречено скептически, поскольку слабый спутник оставался ненаблюдаемым, а его масса должна была быть достаточно велика — сравнимой с массой Сириуса и Проциона, соответственно.

В январе 1862 года Элвин Грэхэм Кларк, юстируя 18-дюймовый рефрактор, самый большой на то время телескоп в мире (Dearborn Telescope), впоследствии поставленный семейной фирмой Кларков в обсерваторию Чикагского университета, обнаружил в непосредственной близости от Сириуса тусклую звёздочку. Это был спутник Сириуса, Сириус B, предсказанный Бесселем[6]. А в 1896 году американский астроном Д. М. Шеберле открыл Процион B, подтвердив тем самым и второе предсказание Бесселя.

В 1915 году американский астроном Уолтер Сидней Адамс измерил спектр Сириуса B. Из измерений следовало, что его температура не ниже, чем у Сириуса A (по современным данным, температура поверхности Сириуса B составляет 25 000 K, а Сириуса A — 10 000 К), что, с учётом его в 10 000 раз меньшей, чем у Сириуса A, светимости указывает на очень малый радиус и, соответственно, высокую плотность — 106 г/см3 (плотность Сириуса ~0,25 г/см3, плотность Солнца ~1,4 г/см3).
"""
print("Длина текста:", len(text))

Длина текста: 1572


In [7]:
from razdel import tokenize
from pymystem3 import Mystem
from itertools import zip_longest

razdel_tokens = [t.text.strip() for t in tokenize(text) if t.text.strip()]

mystem = Mystem()
mystem_tokens = [item["text"].strip() for item in mystem.analyze(text)
                 if item.get("text", "").strip()]

print("Количество токенов razdel:", len(razdel_tokens))
print("Количество токенов Mystem:", len(mystem_tokens))

print("\nПервые 60 токенов razdel:")
print(razdel_tokens[:60])
print("\nПервые 60 токенов Mystem:")
print(mystem_tokens[:60])

print("\nРасхождения после нормализации, первые 40:")
count = 0
for i, (a, b) in enumerate(zip_longest(razdel_tokens, mystem_tokens)):
    if a != b:
        print(f"{i}: razdel={a!r} | mystem={b!r}")
        count += 1
        if count >= 40:
            break

Количество токенов razdel: 292
Количество токенов Mystem: 293

Первые 60 токенов razdel:
['Вторым', 'и', 'третьим', 'открытыми', 'белыми', 'карликами', 'стали', 'Сириус', 'B', 'и', 'Процион', 'B', '.', 'В', '1844', 'году', 'директор', 'Кёнигсбергской', 'обсерватории', 'Фридрих', 'Бессель', ',', 'анализируя', 'данные', 'наблюдений', ',', 'которые', 'велись', 'с', '1755', 'года', ',', 'обнаружил', ',', 'что', 'Сириус', ',', 'ярчайшая', 'звезда', 'земного', 'неба', ',', 'и', 'Процион', 'периодически', ',', 'хотя', 'и', 'весьма', 'слабо', ',', 'отклоняются', 'от', 'прямолинейной', 'траектории', 'движения', 'по', 'небесной', 'сфере', '[']

Первые 60 токенов Mystem:
['Вторым', 'и', 'третьим', 'открытыми', 'белыми', 'карликами', 'стали', 'Сириус', 'B', 'и', 'Процион', 'B', '.', 'В', '1844', 'году', 'директор', 'Кёнигсбергской', 'обсерватории', 'Фридрих', 'Бессель', ',', 'анализируя', 'данные', 'наблюдений', ',', 'которые', 'велись', 'с', '1755', 'года', ',', 'обнаружил', ',', 'что', 'Сириус',

В целом результаты токенизации razdel и Mystem на данном тексте близки. Основное различие связано с обработкой дефисных конструкций: например, razdel сохраняет слово «18-дюймовый» как один токен, а Mystem разделяет его на три части. Поэтому для данного текста razdel удобнее, поскольку он лучше сохраняет целостность составных слов и обозначений, тогда как Mystem полезен для дальнейшего морфологического анализа

## 3. Лемматизация Mystem vs Pymorphy (2 балла)

Лемматизируйте текст с помощью mystem и pymorphy. Найдите различия в лемматизации. Что по вашему работает лучше на приведенном тексте?

Важно: для пайморфи используйте токенизацию из mystem, чтобы исключить влияние токенизации на результат. Анализируйте только значимые различия, а не технические особенности (не сравнивайте скорость работы и удобность интерфейса).

В майстеме убедитесь, что используется дизамбигуация.

In [10]:
import re
import pymorphy3

mystem = Mystem()
morph = pymorphy3.MorphAnalyzer()

def is_word(tok):
    return bool(re.search(r"[А-Яа-яЁёA-Za-z]", tok))

mystem_analysis = mystem.analyze(text)

tokens = []
mystem_lemmas = []
for item in mystem_analysis:
    tok = item.get("text", "")
    if not tok.strip() or not is_word(tok):
        continue
    an = item.get("analysis", [])
    lemma = an[0].get("lex", tok.lower()) if an else tok.lower()
    tokens.append(tok)
    mystem_lemmas.append(lemma.lower())

pymorphy_lemmas = []
for tok in tokens:
    parsed = morph.parse(tok)
    pymorphy_lemmas.append(parsed[0].normal_form.lower() if parsed else tok.lower())

print(f"Сравниваем {len(tokens)} слов\n")
print(f"{'токен':20} | {'Mystem':20} | {'pymorphy':20}")
print("-" * 60)
diff_count = 0
for tok, lm, lp in zip(tokens, mystem_lemmas, pymorphy_lemmas):
    if lm != lp:
        print(f"{tok:20} | {lm:20} | {lp:20}")
        diff_count += 1
print(f"\nВсего расхождений: {diff_count} из {len(tokens)}")

Сравниваем 214 слов

токен                | Mystem               | pymorphy            
------------------------------------------------------------
стали                | становиться          | стать               
Кёнигсбергской       | кенигсбергский       | кёнигсбергский      
данные               | данные               | дать                
обнаружил            | обнаруживать         | обнаружить          
пришёл               | приходить            | прийти              
встречено            | встречать            | встретить           
его                  | его                  | он                  
Грэхэм               | грэхэм               | грэхэма             
поставленный         | поставлять           | поставить           
обнаружил            | обнаруживать         | обнаружить          
звёздочку            | звездочка            | звёздочка           
предсказанный        | предсказывать        | предсказать         
Д                    | д                    | д

Результаты Mystem и Pymorphy могут различаться при определении начальной формы слова, особенно если слово имеет несколько значений или грамматических разборов. Mystem учитывает контекст предложения при выборе наиболее вероятной леммы, тогда как Pymorphy обычно анализирует отдельное слово без учёта его контекстного окружения, поэтому в неоднозначных случаях Mystem может точнее определять значение слова в тексте. Однако нельзя считать один инструмент безусловно лучше: корректность различающихся результатов необходимо оценивать с учётом контекста. Здесь Mystem показал себя несколько лучше

## 4. Лемматизация в SpaCy (2 балла)

С помощью Spacy (модель для русского языка) лемматизируйте тот же текст. Проверьте есть ли различия с Mystem и Pymoprhy.

In [11]:
import spacy

nlp = spacy.load("ru_core_news_sm")
doc = nlp(text)

spacy_pairs = [(t.text, t.lemma_.lower()) for t in doc if t.is_alpha]

print(f"{'токен':20} | {'Mystem':20} | {'pymorphy':20} | {'spacy':20}")
print("-" * 90)
shown = 0
for s_tok, s_lem in spacy_pairs:
    for tok, lm, lp in zip(tokens, mystem_lemmas, pymorphy_lemmas):
        if tok.lower() == s_tok.lower():
            if s_lem != lm or s_lem != lp:
                print(f"{s_tok:20} | {lm:20} | {lp:20} | {s_lem:20}")
                shown += 1
            break
    if shown >= 30:
        break
print(f"\nПоказано расхождений: {shown}")

токен                | Mystem               | pymorphy             | spacy               
------------------------------------------------------------------------------------------
стали                | становиться          | стать                | стать               
Кёнигсбергской       | кенигсбергский       | кёнигсбергский       | кёнигсбергский      
данные               | данные               | дать                 | данные              
которые              | который              | который              | которые             
обнаружил            | обнаруживать         | обнаружить           | обнаружить          
пришёл               | приходить            | прийти               | прийти              
них                  | они                  | они                  | них                 
было                 | быть                 | быть                 | было                
встречено            | встречать            | встретить            | встретить           
его      

SpaCy использует собственную русскую языковую модель, поэтому результаты лемматизации могут отличаться от Mystem и Pymorphy. Особенно важно проверять имена собственные, иностранные названия и неоднозначные словоформы, поскольку правильность леммы зависит от контекста. При сравнении библиотек важно оценивать не только совпадение результатов, но и то, насколько точно каждая из них определяет начальную форму слова с учётом его значения в предложении

## 5*. Поиск фрагемента текста по ключевым словам (2 балла)

Разработайте поиск по неточному описанию эпизода художественного произведения. Сравните поиск по словам, по леммам и один дополнительный способ с использованием spaCy. Найдите релевантный фрагмент и выведите окружающий контекст. Объясните, какие методы помогли и почему.

In [ ]:
from pathlib import Path
import re

novel_path = Path("anna_karenina.txt")
if not novel_path.exists():
    novel_path = Path("/mnt/data/anna_karenina.txt")

novel = novel_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in re.split(r"\n\s*\n", novel) if p.strip()]

print("Символов:", len(novel))
print("Количество абзацев:", len(paragraphs))

Символов: 1710408
Абзацев: 7821


In [19]:
from pathlib import Path
import re
import pymorphy3
import spacy

novel_path = Path("anna_karenina.txt")
if not novel_path.exists():
    novel_path = Path("/mnt/data/anna_karenina.txt")
novel = novel_path.read_text(encoding="utf-8")

p1_starts = [m.start() for m in re.finditer("ЧАСТЬ ПЕРВАЯ", novel)]
p2_starts = [m.start() for m in re.finditer("ЧАСТЬ ВТОРАЯ", novel)]

start = p1_starts[1] if len(p1_starts) > 1 else 0
end = p2_starts[1] if len(p2_starts) > 1 else len(novel)

part1 = novel[start:end]
paragraphs = [p.strip() for p in re.split(r"\n\s*\n", part1) if p.strip()]

print("Позиции 'ЧАСТЬ ПЕРВАЯ':", p1_starts)
print("Позиции 'ЧАСТЬ ВТОРАЯ':", p2_starts)
print("Абзацев для поиска:", len(paragraphs))

#Лемматизация через Pymorphy
morph = pymorphy3.MorphAnalyzer()

def lemmas_set(s):
    out = set()
    for tok in re.findall(r"[А-Яа-яЁёA-Za-z]+", s):
        out.add(morph.parse(tok)[0].normal_form.lower())
    return out

query = "Левин встречается с Кити на катке и сам катается на коньках."

#Метод 1: буквальный поиск
literal_terms = ["левин", "кити", "катке", "катается", "коньках"]
literal_hits = []
for i, p in enumerate(paragraphs):
    low = p.lower()
    score = sum(t in low for t in literal_terms)
    if score:
        literal_hits.append((score, i, p))
literal_hits.sort(key=lambda x: -x[0])

print("\nБуквальный поиск (топ-5)")
for sc, i, p in literal_hits[:5]:
    print(f"\n[абзац {i}, совпадений {sc}]\n{p[:500]}")

#Метод 2: поиск по лемам (Pymorphy)
query_lemmas = {"левин", "кити", "каток", "кататься", "конёк", "конек"}
lemma_hits = []
for i, p in enumerate(paragraphs):
    lw = lemmas_set(p)
    score = len(query_lemmas & lw)
    if score:
        lemma_hits.append((score, i, p))
lemma_hits.sort(key=lambda x: -x[0])

print("\nПоиск по лемам (топ-5)")
for sc, i, p in lemma_hits[:5]:
    print(f"\n[абзац {i}, совпавших лемм {sc}]\n{p[:500]}")

#Метод 3: SpaCy + ключевые леммы
nlp = spacy.load("ru_core_news_sm")

semantic_terms = {"левин", "кити", "каток", "кататься", "конёк", "конек", "лёд", "лед"}
spacy_hits = []
for i, doc in enumerate(nlp.pipe(paragraphs, batch_size=64)):
    lemmas = {t.lemma_.lower() for t in doc if t.is_alpha}
    score = len(semantic_terms & lemmas)
    if "левин" in lemmas and "кити" in lemmas:
        score += 2
    if score:
        spacy_hits.append((score, i, paragraphs[i]))
spacy_hits.sort(key=lambda x: -x[0])

print("\nSpaCy (топ-5)")
for sc, i, p in spacy_hits[:5]:
    print(f"\n[абзац {i}, оценка {sc}]\n{p[:500]}")

best = lemma_hits[0][1] if lemma_hits else None
if best is not None:
    start_ctx, end_ctx = max(0, best - 2), min(len(paragraphs), best + 3)
    print(f"\nКонтекст вокруг абзаца {best} (лучший по лемам)")
    for j in range(start_ctx, end_ctx):
        print(f"\nАбзац {j} \n{paragraphs[j][:1200]}")
else:
    print("\nСовпадений по лемам не найдено - проверьте ключевые леммы.")

Позиции 'ЧАСТЬ ПЕРВАЯ': [501, 782]
Позиции 'ЧАСТЬ ВТОРАЯ': [515, 241756]
Абзацев для поиска: 1180

Буквальный поиск (топ-5)

[абзац 200, совпадений 3]
– Ну, хорошо. Понятно, – сказал Степан Аркадьич. – Так видишь ли: – я бы позвал тебя к себе, но жена не совсем здорова. А вот что: – если ты хочешь их видеть, они, наверное, нынче в Зоологическом саду от четырех до пяти. Кити на коньках катается. Ты поезжай туда, а я заеду, и вместе куда-нибудь обедать.

[абзац 160, совпадений 2]
– Ах да, позвольте вас познакомить, – сказал он. – Мои товарищи: – Филипп Иваныч Никитин, Михаил Станиславич Гриневич, – и обратившись к Левину: – земский деятель, новый земский человек, гимнаст, поднимающий одною рукой пять пудов, скотовод и охотник и мой друг, Константин Дмитрич Левин, брат Сергея Иваныча Кознышева.

[абзац 182, совпадений 2]
Степан Аркадьич, знавший уже давно, что Левин был влюблен в его свояченицу Кити, чуть заметно улыбнулся, и глаза его весело заблестели.

[абзац 211, совпадений 2]
Дома Ле

Помог поиск по лемам, потому что запрос и текст расходятся в словоформах: в описании эпизода в работе «на катке», «катается», «на коньках», а в нужной сцене «к катку», «катка», «кататься», «коньках», «катаетесь». Лемматизация приводит все эти формы к одной основе, поэтому метод находит сцену независимо от падежа и вида глагола. Буквальный поиск по тем же причинам не помог: он сравнивает конкретные строки, а точных форм из запроса в тексте почти нет, поэтому нашёлся только один абзац со случайным совпадением. SpaCy помог частично: он тоже лемматизирует, поэтому попал в сцену, а его дополнительное преимущество - ранжирование по совместному упоминанию Левина и Кити и тематических слов. Однако на модели `sm` лемматизация у SpaCy слабее, чем у Pymorphy. Имена собственные он оставляет в падежной форме, поэтому по точности уступает поиску по лемам.